![Archive file 18, fixed sources v2](assets/archive-window/file-18.png)

# 18 · Comparing forecasts on the fixed cohort

The comparison uses 519 games with complete monthly histories from February 2021 to January 2026. Six fixed model specifications are evaluated over 23 months, matching the distance from the current training cutoff to December 2027. Seasonal naive leads for Indie, while damped seasonal smoothing leads for RPG and Strategy; Action and Massively Multiplayer have much smaller improvements over last value.

The two historical forecast origins share 22 of their 23 target months. These results support a development review, with model selection still provisional.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.evaluate_fixed_cohort import run
predictions, scores, fit_diagnostics, fold_scores = run(ROOT)


![](assets/archive-window/section-divider.png)

## Comparison design

The January and February 2024 cutoffs provide 36 and 37 training months, followed by 23 observed target months. Leads 12–23 cover the same relative position as the 2027 target year in the final forecast. Actual test values are used for scoring after each forecast has been generated.

Last value repeats the final observed month, and seasonal naive repeats the previous annual cycle. Exponential smoothing is tested with a level, a damped trend, and a damped seasonal trend. Gradient boosting uses the existing lag, rolling-average and calendar features, with its parameters held fixed. Recursive features use earlier predictions as the horizon advances.

The cohort and labels were selected using October 2026 knowledge. Historical comparisons are conditional on this retrospectively selected population, and future releases remain outside the target.

In [2]:
comparison_design = pd.DataFrame({
    "Forecast origin": ["January 2024", "February 2024"],
    "Training months": [36, 37],
    "Full test period": ["February 2024–December 2025", "March 2024–January 2026"],
    "Leads 12–23": ["January–December 2025", "February 2025–January 2026"],
})
comparison_design


Forecast origin,Training months,Full test period,Leads 12–23
January 2024,36,February 2024–December 2025,January–December 2025
February 2024,37,March 2024–January 2026,February 2025–January 2026


![](assets/archive-window/section-divider.png)

## Long-horizon errors

MAE describes the average absolute difference in concurrent activity units. WAPE expresses absolute error relative to observed activity within each fold, then averages the two fold percentages. Positive bias indicates underprediction. Category totals overlap, so their errors are assessed separately.

The table shows the lowest average MAE within each category and the last-value reference. Small differences deserve particular care: Action's leading MAE differs from last value by roughly 30 concurrent players, and Massively Multiplayer's by roughly 296.

In [3]:
target_scores = scores.loc[scores.window.eq("Leads 12 to 23")].copy()
leaders = target_scores.sort_values("mae").groupby("category", as_index=False).head(1)
reference = target_scores.loc[target_scores.model.eq("Last value")]
summary = pd.concat([leaders, reference]).drop_duplicates(["category", "model"])
summary = summary.sort_values(["category", "mae"])
summary[["category", "model", "mae", "wape_pct", "mean_error", "wins_vs_last_value"]].round(2)


category,model,mae,wape_pct,mean_error,wins_vs_last_value
Action,Damped trend,87411.56,2.77,230.03,1
Action,Last value,87441.83,2.77,0.84,0
Indie,Seasonal naive,32908.99,4.26,-6619.26,2
Indie,Last value,61647.40,7.98,-19149.18,0
Massively Multiplayer,Damped seasonal,49218.65,6.37,48953.26,1
Massively Multiplayer,Last value,49514.94,6.41,27587.25,0
RPG,Damped seasonal,50004.50,6.09,-19770.80,2
RPG,Last value,106626.11,13.01,-90606.01,0
Strategy,Damped seasonal,50661.16,4.70,-30304.13,2
Strategy,Last value,88856.57,8.24,-35975.73,0


In [4]:
all_models = target_scores.pivot(index="category", columns="model", values="mae")
all_models.round(0).rename_axis(index="Category", columns="Model · MAE")


Model · MAE,Damped seasonal,Damped trend,Gradient boosting,Last value,SES,Seasonal naive
,184233.0,87412.0,91582.0,87442.0,87442.0,121797.0
,75549.0,61607.0,35364.0,61647.0,61647.0,32909.0
,49219.0,49525.0,83704.0,49515.0,49515.0,86951.0
,50005.0,106517.0,74640.0,106626.0,106626.0,101761.0
,50661.0,97562.0,63404.0,88857.0,78248.0,60720.0


![](assets/archive-window/section-divider.png)

## Stability across cutoffs

A lower average error can hide a worse individual fold. The following table retains both cutoffs for the leading models. A recorded win requires more than 0.1% lower MAE than last value, which prevents numerical near-ties from appearing as improvements. Two overlapping cutoffs provide limited evidence of stability.

In [5]:
leader_keys = leaders[["category", "model"]]
leader_folds = fold_scores.loc[fold_scores.window.eq("Leads 12 to 23")].merge(
    leader_keys, on=["category", "model"], validate="many_to_one"
)
leader_folds[["category", "model", "origin", "mae", "skill_vs_last_value_pct"]].assign(origin=lambda frame: frame.origin.dt.strftime("%Y-%m")).round(2)


category,model,origin,mae,skill_vs_last_value_pct
Action,Damped trend,2024-01,83958.58,0.15
Action,Damped trend,2024-02,90864.54,-0.07
Indie,Seasonal naive,2024-01,30769.94,58.30
Indie,Seasonal naive,2024-02,35048.04,29.20
Massively Multiplayer,Damped seasonal,2024-01,46795.83,-27.25
Massively Multiplayer,Damped seasonal,2024-02,51641.47,17.05
RPG,Damped seasonal,2024-01,51434.73,68.47
RPG,Damped seasonal,2024-02,48574.28,3.13
Strategy,Damped seasonal,2024-01,57003.79,45.03
Strategy,Damped seasonal,2024-02,44318.53,40.12


![](assets/archive-window/section-divider.png)

## Fit diagnostics and large misses

Lag creation leaves gradient boosting with 24 or 25 training rows per fold. Its training MAE is an in-sample, one-step fit measure; the future MAE reflects recursive forecasts over a longer horizon. The gap is a warning to inspect model complexity and error patterns, with no tuning justified by these two folds alone. Smoothing convergence and negative-value clipping are recorded in the saved diagnostics.

Large forecast errors are retained for review. Their size alone provides insufficient evidence that an observation is corrupt or should be removed. These are regression forecasts, so false-positive and false-negative counts would require a separately defined event and threshold.

In [6]:
boosting_fit = fit_diagnostics.loc[fit_diagnostics.model.eq("Gradient boosting")]
boosting_future = fold_scores.loc[
    fold_scores.model.eq("Gradient boosting") & fold_scores.window.eq("Leads 12 to 23"),
    ["category", "origin", "mae"],
].rename(columns={"mae": "future_mae_leads_12_23"})
boosting_review = boosting_fit.merge(boosting_future, on=["category", "origin"], validate="one_to_one")
boosting_review[["category", "origin", "training_rows", "training_mae", "future_mae_leads_12_23"]].assign(origin=lambda frame: frame.origin.dt.strftime("%Y-%m")).round(2)


category,origin,training_rows,training_mae,future_mae_leads_12_23
Action,2024-01,24.0,27902.76,88426.01
Action,2024-02,25.0,26452.19,94738.24
Indie,2024-01,24.0,10357.60,39067.98
Indie,2024-02,25.0,10216.72,31659.50
Massively Multiplayer,2024-01,24.0,6439.76,83982.58
Massively Multiplayer,2024-02,25.0,5743.02,83425.53
RPG,2024-01,24.0,26946.12,85506.70
RPG,2024-02,25.0,23269.45,63772.73
Strategy,2024-01,24.0,20192.87,62223.35
Strategy,2024-02,25.0,20790.08,64584.27


In [7]:
largest_misses = predictions.assign(absolute_error=lambda frame: frame.error.abs())
largest_misses = largest_misses.sort_values("absolute_error", ascending=False)
largest_misses[["category", "model", "origin", "month", "lead", "actual", "prediction", "error"]].head(6).assign(origin=lambda frame: frame.origin.dt.strftime("%Y-%m"), month=lambda frame: frame.month.dt.strftime("%Y-%m")).round(2)


category,model,origin,month,lead,actual,prediction,error
Action,Damped seasonal,2024-02,2025-11,21.0,3140854.22,2751551.18,389303.04
Action,Damped seasonal,2024-02,2024-06,4.0,3339613.00,2954314.77,385298.23
RPG,Seasonal naive,2024-02,2025-08,18.0,825840.56,1184299.36,-358458.80
RPG,Seasonal naive,2024-01,2025-08,19.0,825840.56,1184299.36,-358458.80
Strategy,Seasonal naive,2024-02,2024-08,6.0,1072189.83,1430016.43,-357826.60
Strategy,Seasonal naive,2024-01,2024-08,7.0,1072189.83,1430016.43,-357826.60


![](assets/archive-window/section-divider.png)

## Model comparison findings

Seasonal naive is a useful candidate for Indie, while damped seasonal smoothing deserves closer review for RPG and Strategy. The small average improvements for Action and Massively Multiplayer provide limited reason to replace a simple reference. Gradient boosting remains a comparison model and requires care given the small training sample.

Notebook 19 reviews the leading models' largest errors, cutoff sensitivity and category composition; Notebook 20 records the resulting provisional policy. No new sources, parameter search or final 2027 forecasts are introduced here.